In [1]:
import pandas as pd
import numpy as np

In [2]:
customer = pd.read_csv("customer.csv")
demographic = pd.read_csv("demographic.csv")
address = pd.read_csv("address.csv")
termination = pd.read_csv("termination.csv")
 

In [3]:
print("Customer:", customer.shape)
print("Demographic:",demographic.shape)
print("Address:",address.shape)
print("Termination:",termination.shape)

Customer: (2280321, 8)
Demographic: (2112579, 9)
Address: (1536673, 7)
Termination: (269259, 2)


In [4]:
# Merging all the datasets -
df = customer.merge(
    demographic,
    on="INDIVIDUAL_ID",
    how="left"
    )

In [5]:
df = df.merge(
    address,
    on="ADDRESS_ID",
    how="left"
    )
df = df.merge(
    termination,
    on="INDIVIDUAL_ID",
    how="left"
    )

In [6]:
print(df.shape)       # shape of the merged dataset
print(df.head())

(2280321, 23)
   INDIVIDUAL_ID    ADDRESS_ID  CURR_ANN_AMT  DAYS_TENURE CUST_ORIG_DATE  \
0   2.213000e+11  5.213000e+11    818.877997       1454.0     2018-12-09   
1   2.213001e+11  5.213001e+11    974.199182       1795.0     2018-01-02   
2   2.213007e+11  5.213002e+11    967.375112       4818.0     2009-09-23   
3   2.213016e+11  5.213006e+11    992.409561        130.0     2022-07-25   
4   2.213016e+11  5.213006e+11    784.633494       5896.0     2006-10-11   

   AGE_IN_YEARS DATE_OF_BIRTH SOCIAL_SECURITY_NUMBER    INCOME  HAS_CHILDREN  \
0        44.474    1978-06-23            608-XX-7640   22500.0           1.0   
1        72.559    1950-05-30            342-XX-6908   27500.0           0.0   
2        55.444    1967-07-07            240-XX-9224   42500.0           0.0   
3        53.558    1969-05-25            775-XX-6249  125000.0           1.0   
4        50.220    1972-09-25            629-XX-7298   87500.0           1.0   

   ...  HOME_OWNER COLLEGE_DEGREE GOOD_CREDIT   

In [7]:
# converting dates
df["CUST_ORIG_DATE"] = pd.to_datetime(
    df["CUST_ORIG_DATE"],
    errors="coerce"
)

df["ACCT_SUSPD_DATE"] = pd.to_datetime(
    df["ACCT_SUSPD_DATE"],
    errors="coerce"
)

In [8]:
# creating "event" column 
df["event"] = df["ACCT_SUSPD_DATE"].notna().astype(int)

In [9]:
# Creating observation end date
snapshot_date = pd.Timestamp("2022-12-02")

df["observation_end_date"] = (df["ACCT_SUSPD_DATE"].fillna(snapshot_date))

In [10]:
# Renaming the starting date
df = df.rename(
    columns={
        "CUST_ORIG_DATE": "starting_date"
    }
)

In [11]:
# calculating duration 
df["duration_days"] = (
   df["observation_end_date"] - df["starting_date"]
).dt.days

In [12]:
#snapshot_date = pd.Timestamp("2022-12-02")

#df["days_until_snapshot"] = (
#    snapshot_date - df["CUST_ORIG_DATE"]
#).dt.days

In [13]:
'''df["duration_days"] = df["days_until_suspension"]

df.loc[
    df["event"] == 0,
    "duration_days"
] = df["days_until_snapshot"]'''

'df["duration_days"] = df["days_until_suspension"]\n\ndf.loc[\n    df["event"] == 0,\n    "duration_days"\n] = df["days_until_snapshot"]'

In [14]:
# removing invalid records
df = df[df["INDIVIDUAL_ID"].notna()]
df = df[df["starting_date"].notna()]
df = df[df["duration_days"] > 0]
df = df[df["CURR_ANN_AMT"].notna()]
df = df[df["CURR_ANN_AMT"] >= 0]

In [15]:
# selecting required columns only
df = df[
    [
        "INDIVIDUAL_ID",
        "starting_date",
        "ACCT_SUSPD_DATE",
        "observation_end_date",
        "duration_days",
        "event",
        "AGE_IN_YEARS",
        "INCOME",
        "CURR_ANN_AMT",
        "HAS_CHILDREN",
        "MARITAL_STATUS",
        "HOME_OWNER",
        "COLLEGE_DEGREE",
        "GOOD_CREDIT",
        "COUNTY"
    ]
].copy()


In [16]:
# Renaming the columns
df = df.rename(
    columns={
        "INDIVIDUAL_ID": "customer_id",
        "ACCT_SUSPD_DATE": "suspension_date",
        "AGE_IN_YEARS": "age",
        "INCOME": "income",
        "CURR_ANN_AMT": "annual_premium",
        "HAS_CHILDREN": "has_children",
        "MARITAL_STATUS": "marital_status",
        "HOME_OWNER": "home_owner",
        "COLLEGE_DEGREE": "college_degree",
        "GOOD_CREDIT": "good_credit",
        "COUNTY": "county"
    }
)

In [17]:
df["age"]

0          44.474
1          72.559
2          55.444
4          50.220
5          32.641
            ...  
2280316    52.389
2280317    37.388
2280318    55.444
2280319       NaN
2280320    63.639
Name: age, Length: 2197537, dtype: float64

In [18]:
# Handling missing values 
df["marital_status"] = df["marital_status"].fillna("Unknown")
df["county"] = df["county"].fillna("Unknown")

In [19]:
# checking the final dataset
print("The final dataset shape :",df.shape)
print("The first 5 rows :\n", df.head())     
print("Missing values :\n", df.isna().sum())


The final dataset shape : (2197537, 15)
The first 5 rows :
     customer_id starting_date suspension_date observation_end_date  \
0  2.213000e+11    2018-12-09             NaT           2022-12-02   
1  2.213001e+11    2018-01-02             NaT           2022-12-02   
2  2.213007e+11    2009-09-23             NaT           2022-12-02   
4  2.213016e+11    2006-10-11             NaT           2022-12-02   
5  2.213027e+11    2021-08-05             NaT           2022-12-02   

   duration_days  event     age   income  annual_premium  has_children  \
0           1454      0  44.474  22500.0      818.877997           1.0   
1           1795      0  72.559  27500.0      974.199182           0.0   
2           4818      0  55.444  42500.0      967.375112           0.0   
4           5896      0  50.220  87500.0      784.633494           1.0   
5            484      0  32.641  52500.0      909.916163           0.0   

  marital_status  home_owner  college_degree  good_credit   county  
0    

In [20]:
# checking the event distribution 
print(df["event"].value_counts())

event
0    2011000
1     186537
Name: count, dtype: int64


In [21]:
# calculating event rate
print(df["event"].mean() * 100)

8.488457759755581


In [22]:
# saving the final dataset
df.to_csv("dataset_file.csv",index=False)
print("Dataset saved successfully!")

Dataset saved successfully!
